# Обучение классификатора (Этап 4)

Два режима:
1. **Random Forest** на морфометрических метриках из metadata.csv (baseline)
2. **MLP** на latent vector (256) + морфометрия (5 признаков) → normal/anomaly

**Предусловие:** `best_autoencoder.pt` (от этапа 1) — нужен для MLP-режима.

**Время обучения:**
- RF: ~10 секунд
- MLP: ~5-10 минут на T4

In [ ]:
# 1. Монтируем Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Скачиваем актуальный код
%cd /content
!rm -rf TriView3D
!git clone https://github.com/3x6dll9ff/TriView3D.git
%cd TriView3D

In [ ]:
# 3. Зависимости
!pip install -q -r requirements.txt

In [ ]:
# 4. Проверка GPU
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('CPU only — для MLP хватит, RF и так без GPU')

In [ ]:
# 5. Подготовка: данные + базовая модель
import os, shutil, glob

# Распаковка данных
!rm -rf /content/data
!unzip -q /content/drive/MyDrive/data_processed.zip -d /content/

# Находим директорию с данными
csv_files = glob.glob('/content/**/metadata.csv', recursive=True)
if csv_files:
    DATA_PATH = os.path.dirname(csv_files[0])
    os.environ['DATA_PATH'] = DATA_PATH
    print(f'Данные найдены: {DATA_PATH}')
else:
    raise FileNotFoundError('metadata.csv не найден!')

# Копируем autoencoder для MLP-режима
os.makedirs('results', exist_ok=True)

ae_candidates = [
    '/content/best_autoencoder.pt',
    '/content/drive/MyDrive/best_autoencoder.pt',
]
ae_found = False
for src in ae_candidates:
    if os.path.exists(src):
        shutil.copy2(src, 'results/best_autoencoder.pt')
        print(f'Autoencoder скопирован: {src}')
        ae_found = True
        break

if not ae_found:
    print('⚠️ best_autoencoder.pt не найден — загрузите вручную:')
    from google.colab import files
    uploaded = files.upload()
    for filename in uploaded:
        with open(os.path.join('results', filename), 'wb') as f:
            f.write(uploaded[filename])
        print(f'Сохранено: results/{filename}')

In [ ]:
# 6a. Random Forest (baseline — ~10 секунд)
!python3 src/train_classifier.py \
    --data_dir "$DATA_PATH" \
    --mode rf

In [ ]:
# 6b. MLP классификатор на latent + морфометрия (~5-10 мин)
!python3 src/train_classifier.py \
    --data_dir "$DATA_PATH" \
    --mode latent \
    --autoencoder results/best_autoencoder.pt \
    --input_mode tri \
    --epochs 40 \
    --batch_size 16 \
    --lr 1e-3

In [ ]:
# 7. Проверяем результаты
import json, os

for name in ['rf_results.json', 'mlp_results.json']:
    path = f'results/metrics/{name}'
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        print(f'\n=== {name} ===')
        for k, v in data.items():
            print(f'  {k}: {v}')

# Проверяем наличие сохранённых моделей
for model_file in ['best_classifier.pt', 'best_rf_classifier.pkl']:
    path = f'results/{model_file}'
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'\n✅ {model_file}: {size_mb:.2f} MB')
    else:
        print(f'\n❌ {model_file} не найден')

In [ ]:
# 8. Скачиваем модели
from google.colab import files
import os

for f_name in ['results/best_classifier.pt', 'results/best_rf_classifier.pkl',
               'results/metrics/rf_results.json', 'results/metrics/mlp_results.json']:
    if os.path.exists(f_name):
        files.download(f_name)
        print(f'Скачан: {f_name}')
    else:
        print(f'Пропущен: {f_name}')